# CS221 ViHOS — Huấn luyện PhoBERT / ViSoBERT Token Classification (Google Colab GPU)

**Đề tài:** Phát hiện đoạn ngôn từ thù ghét và xúc phạm trong tiếng Việt (ViHOS Spans Detection)

**Mục tiêu Notebook:**
1. Tận dụng GPU T4 miễn phí trên Google Colab để huấn luyện **PhoBERT (`vinai/phobert-base-v2`)** (Stage B) hoặc **ViSoBERT (`uitnlp/visobert`)** (Stage C) trong ~5 phút.
2. Dự đoán toàn diện trên tập **Dev** (1,106 câu).
3. Đo lường đầy đủ theo **5 Trụ cột đánh giá** (SemEval-2021 Char Macro-F1, Token F1, Post-level, Phân rã lỗi ranh giới H3).
4. Tự động xuất file kết quả `predictions.csv` để bạn tải về máy ném vào thư mục `experiments/predictions/` của repo!

## 1. Cài đặt thư viện & Kiểm tra GPU

In [1]:
# Cài đặt các thư viện cần thiết
!pip install -q transformers datasets accelerate seqeval scikit-learn pandas

import torch
print("PyTorch Version:", torch.__version__)
print("CUDA Available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU Device:", torch.cuda.get_device_name(0))
    print("VRAM:", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 2), "GB")
else:
    print("CẢNH BÁO: Bạn chưa bật GPU! Hãy vào menu: Runtime -> Change runtime type -> Chọn T4 GPU")

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
PyTorch Version: 2.11.0+cu130
CUDA Available: True
GPU Device: Tesla T4
VRAM: 15.64 GB


## 2. Nạp dữ liệu ViHOS chính thức

In [2]:
# Clone dataset chính thức từ HuggingFace
import os
if not os.path.exists("DATA/ViHOS_official"):
    !git clone https://huggingface.co/datasets/phusroyal/ViHOS.git DATA/ViHOS_official
    print("Đã tải thành công ViHOS official!")
else:
    print("Dữ liệu đã có sẵn trong DATA/ViHOS_official")

Cloning into 'DATA/ViHOS_official'...
remote: Enumerating objects: 229, done.
remote: Total 229 (delta 0), reused 0 (delta 0), pack-reused 229 (from 1)
Receiving objects: 100% (229/229), 2.57 MiB | 11.93 MiB/s, done.
Resolving deltas: 100% (125/125), done.
Đã tải thành công ViHOS official!


## 3. Cấu hình Thử nghiệm

In [3]:
# CHỌN MÔ HÌNH VÀ CẤU HÌNH Ở ĐÂY:
# Stage B (PhoBERT): 'vinai/phobert-base-v2'
# Stage C (ViSoBERT): 'uitnlp/visobert'
MODEL_NAME = "vinai/phobert-base-v2"
MODE = "syllable"          # 'syllable' hoặc 'word'
BATCH_SIZE = 16
LEARNING_RATE = 3e-5
NUM_EPOCHS = 3
MAX_LENGTH = 160
SEED = 42

LABEL2ID = {"O": 0, "B-T": 1, "I-T": 2}
ID2LABEL = {0: "O", 1: "B-T", 2: "I-T"}

print(f"Cấu hình: Model={MODEL_NAME} | Mode={MODE} | Epochs={NUM_EPOCHS} | BatchSize={BATCH_SIZE}")

Cấu hình: Model=vinai/phobert-base-v2 | Mode=syllable | Epochs=3 | BatchSize=16


## 4. Xây dựng Pipeline Nạp & Tiền xử lý Dữ liệu

In [4]:
import ast
import pandas as pd
from typing import List, Set, Tuple, Dict, Any
from torch.utils.data import Dataset

class ViHOSSample:
    def __init__(self, sentence_id: int, content: str, tokens: List[str], tags: List[str], gold_char_offsets: Set[int]):
        self.sentence_id = sentence_id
        self.content = content
        self.tokens = tokens
        self.tags = tags
        self.gold_char_offsets = gold_char_offsets
        self.is_toxic = 1 if ("B-T" in tags or len(gold_char_offsets) > 0) else 0

def align_tokens_to_char_spans(content: str, tokens: List[str]) -> List[Tuple[int, int]]:
    token_spans = []
    curr_idx = 0
    content_lower = content.lower()
    for token in tokens:
        tok_clean = str(token).strip()
        idx = content.find(tok_clean, curr_idx)
        if idx == -1:
            idx = content_lower.find(tok_clean.lower(), curr_idx)
        if idx != -1:
            start, end = idx, idx + len(tok_clean)
            token_spans.append((start, end))
            curr_idx = end
        else:
            start, end = curr_idx, min(curr_idx + len(tok_clean), len(content))
            token_spans.append((start, end))
            curr_idx = end
    return token_spans

def bio_to_char_offsets(tokens: List[str], tags: List[str], content: str) -> Set[int]:
    token_spans = align_tokens_to_char_spans(content, tokens)
    char_offsets = set()
    prev_end, prev_tag = None, None
    for (start, end), tag in zip(token_spans, tags):
        if tag in ("B-T", "I-T"):
            if tag == "I-T" and prev_tag in ("B-T", "I-T") and prev_end is not None and prev_end < start:
                for c in range(prev_end, start):
                    char_offsets.add(c)
            for c in range(start, end):
                char_offsets.add(c)
        prev_end, prev_tag = end, tag
    return char_offsets

def load_vihos_split(split: str = "train", mode: str = "syllable", data_dir: str = "DATA/ViHOS_official") -> List[ViHOSSample]:
    if mode == "syllable":
        bio_file = f"{data_dir}/train_sequence_labeling/syllable/{split}_BIO_syllable.csv"
    else:
        bio_file = f"{data_dir}/train_sequence_labeling/word/{split}_BIO_Word.csv"

    span_file = f"{data_dir}/train_span_extraction/{split}.csv" if split in ("train", "dev") else f"{data_dir}/test/test.csv"

    df_span = pd.read_csv(span_file)
    df_bio = pd.read_csv(bio_file)
    grouped_bio = df_bio.groupby("sentence_id")

    samples = []
    for sid in range(len(df_span)):
        span_row = df_span.iloc[sid]
        content = str(span_row["content"]) if pd.notna(span_row["content"]) else ""
        raw_spans = span_row.get("index_spans", "[]")
        gold_char_offsets = set(ast.literal_eval(str(raw_spans))) if pd.notna(raw_spans) and str(raw_spans).strip() not in ("", "nan") else set()

        if sid in grouped_bio.groups:
            grp = grouped_bio.get_group(sid)
            tokens = [str(w) if pd.notna(w) else "" for w in grp["Word"].tolist()]
            tags = [str(t) if pd.notna(t) else "O" for t in grp["Tag"].tolist()]
        else:
            tokens = content.split()
            tags = ["O"] * len(tokens)

        samples.append(ViHOSSample(sid, content, tokens, tags, gold_char_offsets))
    return samples

train_samples = load_vihos_split("train", mode=MODE)
dev_samples = load_vihos_split("dev", mode=MODE)
print(f"Đã nạp: Train = {len(train_samples):,} câu | Dev = {len(dev_samples):,} câu")

Đã nạp: Train = 8,844 câu | Dev = 1,106 câu


## 5. Chuẩn bị PyTorch Dataset với Tokenizer Căn Chỉnh Subword

In [5]:
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

class ViHOSTransformerDataset(Dataset):
    def __init__(self, samples: List[ViHOSSample], tokenizer, max_length: int = 160):
        self.samples = samples
        self.tokenizer = tokenizer
        self.max_length = max_length
        self.features = [self._prepare_feature(s) for s in samples]

    def _prepare_feature(self, sample: ViHOSSample):
        tokens = sample.tokens
        tags = sample.tags

        bos = self.tokenizer.bos_token or self.tokenizer.cls_token
        eos = self.tokenizer.eos_token or self.tokenizer.sep_token

        subwords = [bos]
        subword_labels = [-100]
        first_subword_indices = []

        for word, tag in zip(tokens, tags):
            w_tokens = self.tokenizer.tokenize(str(word))
            if not w_tokens:
                w_tokens = [self.tokenizer.unk_token]
            first_subword_indices.append(len(subwords))
            for i, subw in enumerate(w_tokens):
                subwords.append(subw)
                subword_labels.append(LABEL2ID.get(tag, 0) if i == 0 else -100)

        subwords.append(eos)
        subword_labels.append(-100)

        if len(subwords) > self.max_length:
            subwords = subwords[:self.max_length - 1] + [eos]
            subword_labels = subword_labels[:self.max_length - 1] + [-100]
            first_subword_indices = [idx for idx in first_subword_indices if idx < self.max_length - 1]

        input_ids = self.tokenizer.convert_tokens_to_ids(subwords)
        attention_mask = [1] * len(input_ids)

        pad_len = self.max_length - len(input_ids)
        pad_id = self.tokenizer.pad_token_id if self.tokenizer.pad_token_id is not None else 1
        input_ids += [pad_id] * pad_len
        attention_mask += [0] * pad_len
        subword_labels += [-100] * pad_len

        return {
            "input_ids": torch.tensor(input_ids, dtype=torch.long),
            "attention_mask": torch.tensor(attention_mask, dtype=torch.long),
            "labels": torch.tensor(subword_labels, dtype=torch.long),
            "first_subword_indices": first_subword_indices,
            "sentence_id": sample.sentence_id
        }

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        return self.features[idx]

train_dataset = ViHOSTransformerDataset(train_samples, tokenizer, max_length=MAX_LENGTH)
dev_dataset = ViHOSTransformerDataset(dev_samples, tokenizer, max_length=MAX_LENGTH)
print("Đã chuẩn bị xong Dataset cho Transformer!")

config.json:   0%|          | 0.00/678 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/895k [00:00<?, ?B/s]

bpe.codes:   0%|          | 0.00/1.14M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/3.13M [00:00<?, ?B/s]

Đã chuẩn bị xong Dataset cho Transformer!


## 6. Huấn luyện Mô hình với HuggingFace Trainer

In [6]:
import numpy as np
from seqeval.metrics import f1_score as seq_f1_score
from transformers import AutoModelForTokenClassification, TrainingArguments, Trainer

def compute_metrics(p):
    predictions, labels = p
    predictions = np.argmax(predictions, axis=2)

    true_labels = [
        [ID2LABEL[l] for (p_idx, l) in zip(pred, lab) if l != -100]
        for pred, lab in zip(predictions, labels)
    ]
    true_predictions = [
        [ID2LABEL[p_idx] for (p_idx, l) in zip(pred, lab) if l != -100]
        for pred, lab in zip(predictions, labels)
    ]
    return {"f1": seq_f1_score(true_labels, true_predictions)}

model = AutoModelForTokenClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(LABEL2ID),
    id2label=ID2LABEL,
    label2id=LABEL2ID
)

output_dir = f"checkpoints_{MODEL_NAME.replace('/', '_')}"

training_args = TrainingArguments(
    output_dir=output_dir,
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=LEARNING_RATE,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    num_train_epochs=NUM_EPOCHS,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
    fp16=torch.cuda.is_available(),
    logging_steps=50,
    seed=SEED,
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=dev_dataset,
    compute_metrics=compute_metrics
)

print("BẮT ĐẦU HUẤN LUYỆN (Thời gian dự kiến: ~5 phút)...!")
trainer.train()

pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  540MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] RobertaForTokenClassification LOAD REPORT from: vinai/phobert-base-v2
Key                       | Status     | 
--------------------------+------------+-
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.dense.bias        | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
classifier.weight         | MISSING    | 
classifier.bias           | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


BẮT ĐẦU HUẤN LUYỆN (Thời gian dự kiến: ~5 phút)...!


model.safetensors: reconstructing file:   0%|          |  0.00B /  540MB            

... (2 outputs left out)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=1659, training_loss=0.3017090398766608, metrics={'train_runtime': 266.3699, 'train_samples_per_second': 99.606, 'train_steps_per_second': 6.228, 'total_flos': 2166496321271040.0, 'train_loss': 0.3017090398766608, 'epoch': 3.0})

## 7. Dự đoán & Đo lường Toàn diện 5 Trụ Cột trên Tập Dev

In [7]:
# Dự đoán trên toàn bộ Dev
raw_pred = trainer.predict(dev_dataset)
pred_logits = raw_pred.predictions
pred_ids = np.argmax(pred_logits, axis=2)

# Ánh xạ ngược từ subwords về tokens gốc
dev_pred_tags = []
for idx, sample in enumerate(dev_samples):
    feat = dev_dataset.features[idx]
    indices = feat["first_subword_indices"]
    seq_pred_ids = pred_ids[idx]

    tags = []
    for tok_idx, subw_idx in enumerate(indices):
        if subw_idx < len(seq_pred_ids):
            tags.append(ID2LABEL.get(seq_pred_ids[subw_idx], "O"))
        else:
            tags.append("O")

    while len(tags) < len(sample.tokens):
        tags.append("O")
    dev_pred_tags.append(tags[:len(sample.tokens)])

# Chuyển thành character offsets & post-level prediction
pred_char_offsets_list = []
records = []

for sample, tags in zip(dev_samples, dev_pred_tags):
    offsets = bio_to_char_offsets(sample.tokens, tags, sample.content)
    pred_char_offsets_list.append(offsets)
    is_pred_toxic = 1 if ("B-T" in tags or len(offsets) > 0) else 0

    records.append({
        "sentence_id": sample.sentence_id,
        "content": sample.content,
        "gold_tags": " ".join(sample.tags),
        "pred_tags": " ".join(tags),
        "gold_char_offsets": str(sorted(list(sample.gold_char_offsets))),
        "pred_char_offsets": str(sorted(list(offsets))),
        "gold_toxic": sample.is_toxic,
        "pred_toxic": is_pred_toxic
    })

model_tag = "phobert" if "phobert" in MODEL_NAME.lower() else "visobert"
pred_filename = f"{model_tag}_{MODE}_dev_predictions.csv"
pred_df = pd.DataFrame(records)
pred_df.to_csv(pred_filename, index=False, encoding="utf-8")
print(f"-> ĐÃ XUẤT FILE PREDICTIONS: {pred_filename}")

-> ĐÃ XUẤT FILE PREDICTIONS: phobert_syllable_dev_predictions.csv


## 8. Bảng Tổng Hợp Kết Quả Đánh Giá (5 Trụ Cột)

In [8]:
# Tính toán các metrics
def f1_per_char_span(gold, pred):
    if not gold and not pred: return 1.0, 1.0, 1.0
    if not gold or not pred: return 0.0, 0.0, 0.0
    ov = len(gold.intersection(pred))
    p, r = ov / len(pred), ov / len(gold)
    return p, r, (2 * p * r / (p + r)) if (p + r) > 0 else 0.0

gold_char_offsets_list = [s.gold_char_offsets for s in dev_samples]
ps, rs, f1s = [], [], []
single_f1s, multi_f1s = [], []

for gold, pred in zip(gold_char_offsets_list, pred_char_offsets_list):
    p, r, f = f1_per_char_span(gold, pred)
    ps.append(p); rs.append(r); f1s.append(f)
    if gold:
        sorted_g = sorted(list(gold))
        nspans = 1 + sum(1 for i in range(1, len(sorted_g)) if sorted_g[i] != sorted_g[i-1] + 1)
        if nspans == 1: single_f1s.append(f)
        else: multi_f1s.append(f)

from sklearn.metrics import accuracy_score, f1_score as sk_f1
post_acc = accuracy_score([s.is_toxic for s in dev_samples], pred_df["pred_toxic"])
post_f1 = sk_f1([s.is_toxic for s in dev_samples], pred_df["pred_toxic"], average="binary")

# Phân rã lỗi ranh giới (H3)
exact, miss, spurious, under, over, shifted, clean_corr = 0, 0, 0, 0, 0, 0, 0
for gold, pred in zip(gold_char_offsets_list, pred_char_offsets_list):
    if not gold and not pred: clean_corr += 1
    elif gold and not pred: miss += 1
    elif not gold and pred: spurious += 1
    elif gold == pred: exact += 1
    else:
        inter = gold.intersection(pred)
        if len(inter) == 0: miss += 1
        elif pred.issubset(gold): under += 1
        elif gold.issubset(pred): over += 1
        elif min(gold) != min(pred) and max(gold) != max(pred): shifted += 1
        elif len(pred) < len(gold): under += 1
        else: over += 1

tot = len(dev_samples)
print("=" * 75)
print(f"KẾT QUẢ ĐÁNH GIÁ MÔ HÌNH: {MODEL_NAME.upper()}")
print("=" * 75)
print(f"[Trụ cột 1 - Span & Plausibility]")
print(f"  • Character Macro-F1 (SemEval):  {np.mean(f1s):.4f}")
print(f"  • Char Precision / Recall:       {np.mean(ps):.4f} / {np.mean(rs):.4f}")
print(f"  • Char F1 Single-Span:           {np.mean(single_f1s):.4f}")
print(f"  • Char F1 Multiple-Span:         {np.mean(multi_f1s):.4f}")
print(f"\n[Trụ cột 2 - Post-level Classification]")
print(f"  • Post Accuracy:                 {post_acc:.4f}")
print(f"  • Post Binary-F1 (Toxic/Clean):  {post_f1:.4f}")
print(f"\n[Trụ cột 5 - Phân rã lỗi ranh giới (H3 Audit)]")
print(f"  • Exact Match:      {exact/tot*100:5.2f}%")
print(f"  • Under-span:       {under/tot*100:5.2f}%")
print(f"  • Over-span:        {over/tot*100:5.2f}%")
print(f"  • Shifted Boundary: {shifted/tot*100:5.2f}%")
print(f"  • Complete Miss:    {miss/tot*100:5.2f}%")
print(f"  • Spurious Span:    {spurious/tot*100:5.2f}%")
print(f"  • Clean Correct:    {clean_corr/tot*100:5.2f}%")
print("=" * 75)

KẾT QUẢ ĐÁNH GIÁ MÔ HÌNH: VINAI/PHOBERT-BASE-V2
[Trụ cột 1 - Span & Plausibility]
  • Character Macro-F1 (SemEval):  0.8152
  • Char Precision / Recall:       0.8498 / 0.8234
  • Char F1 Single-Span:           0.7247
  • Char F1 Multiple-Span:         0.6937

[Trụ cột 2 - Post-level Classification]
  • Post Accuracy:                 0.9141
  • Post Binary-F1 (Toxic/Clean):  0.9115

[Trụ cột 5 - Phân rã lỗi ranh giới (H3 Audit)]
  • Exact Match:      14.92%
  • Under-span:       15.10%
  • Over-span:        11.84%
  • Shifted Boundary:  2.08%
  • Complete Miss:     4.61%
  • Spurious Span:     4.25%
  • Clean Correct:    47.20%


## 9. Tải File Predictions về Máy Cá Nhân
Chạy ô code dưới đây để tải file CSV kết quả về máy. Sau đó chỉ cần ném file này vào thư mục `experiments/predictions/` trong repo của bạn!

In [9]:
from google.colab import files
print(f"Đang tải {pred_filename} về máy của bạn...")
files.download(pred_filename)

Đang tải phobert_syllable_dev_predictions.csv về máy của bạn...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>